# Step 2: V8 Allocation & Basin Screening

## Match precomputed 150-year basin resources to V8 national CO2 storage demand paths

This notebook runs and post-processes the full V8 screening workflow across:

- 8 growth scenarios: reference, minimum, maximum, growth10, us1gt, policy, ipcc_low, ipcc_high
- 10 study regions: Australia, Brazil, Canada, China, EU, Indonesia, Middle East, Thailand, UK, US
- 2 growth models: Logistic and Gompertz
- 2 allocation strategies: descending and ascending
- 3 portfolio checkpoints: 2050, 2100, 2179

The allocation horizon is 2030-2179. Each ordering has 160 requested scenario-region-model combinations; 150 have an available V8 central growth path and are carried into the whole-period screening tables.

### Parameters

| Parameter | Value | Description |
|-----------|------:|-------------|
| `INJ_DURATION_YR` | 150 | Matches Step 1 resource matrix |
| `ALLOCATION_DURATION_YR` | 150 | Full planning horizon |
| `LAST_ALLOC_YEAR` | 2179 | 2030 + 150 - 1 |
| `STORAGE_PERIOD_STEP_YR` | 10 | Period increment |
| `GROWTH_PATHWAY_LABEL` | v8_fixedC_deterministic | Fixed-C deterministic central pathway from V8 |
| `ALLOCATION_ORDERS` | descend, ascend | Both strategies evaluated |

### Inputs from Step 1

- `output/step1_precompute/precompute/basin_period_resource_long.csv`
- `output/step1_precompute/precompute/basin_metadata.csv`

### Status taxonomy

| Status | Meaning |
|--------|---------|
| `ok` | Allocation ran successfully |
| `no_basin_identified` | No matched basins for the requested study region |
| `no_growth_path` | No V8 fixed-C deterministic path for this study region under this scenario/model |
| `allocation_error` | Unexpected error during allocation |

In [1]:
from __future__ import annotations

import math
import sys
import warnings
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=FutureWarning)

# Project paths
NB_DIR = Path.cwd()
SCREENING_ROOT = NB_DIR.parent
CODE_ROOT = SCREENING_ROOT / 'code'

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from co2block_py.allocation import AllocationConfig, run_allocation_workflow
from co2block_py.screening import resolve_region_nos, slugify

# Input paths
GROWTH_MODEL_ROOT = SCREENING_ROOT.parent / '01_growth_model'
GROWTH_TIMESERIES = GROWTH_MODEL_ROOT / 'output' / 'v8_2026-06-01' / 'timeseries_central.csv'
BASIN_FILE = SCREENING_ROOT / 'input' / 'basin_data' / 'Global.xlsx'
PRECOMPUTE_DIR = SCREENING_ROOT / 'output' / 'step1_precompute' / 'precompute'

# Output root
OUTPUT_ROOT = SCREENING_ROOT / 'output' / 'step2_screening'

# Allocation parameters (150-year, matches Step 1 & MATLAB Algorithm2)
INJ_DURATION_YR        = 150
ALLOCATION_DURATION_YR = 150
STORAGE_PERIOD_STEP_YR = 10
NR_DIST                = 100
MAX_Q_MT_YR            = 20.0
MIN_Q_MT_YR            = 1.0
SCREENING_SCOPE        = 'country'

LAST_ALLOC_YEAR = 2030 + ALLOCATION_DURATION_YR - 1  # 2179
CHECKPOINTS = [2050, 2100, LAST_ALLOC_YEAR]

# Screening dimensions
GROWTH_PATHWAY_LABEL = 'v8_fixedC_deterministic'

SCENARIOS = [
    'reference', 'minimum', 'maximum', 'growth10',
    'us1gt', 'policy', 'ipcc_low', 'ipcc_high',
]

# V8 (2026-06-01) active study pool. EU is Europe without UK.
SELECTED_CASES = [
    'UK', 'US', 'EU', 'China', 'Middle East',
    'Australia', 'Canada', 'Indonesia', 'Thailand', 'Brazil',
]

MODELS = ['Logistic', 'Gompertz']

ALLOCATION_ORDERS = ['descend', 'ascend']
ORDER_DIR_MAP = {'descend': 'descending', 'ascend': 'ascending'}

print(f'Growth:      {GROWTH_PATHWAY_LABEL}')
print(f'Source:      {GROWTH_TIMESERIES}')
print(f'Scenarios:   {SCENARIOS}')
print(f'Countries:   {SELECTED_CASES}')
print(f'Models:      {MODELS}')
print(f'Orders:      {ALLOCATION_ORDERS}')
print(f'Checkpoints: {CHECKPOINTS}')
print(f'Horizon:     2030 – {LAST_ALLOC_YEAR} ({ALLOCATION_DURATION_YR} yr)')


Growth:      v8_fixedC_deterministic
Source:      /Users/xg320/Desktop/IC-Sam-works/paper/Iman-2026/01_growth_model/output/v8_2026-06-01/timeseries_central.csv
Scenarios:   ['reference', 'minimum', 'maximum', 'growth10', 'us1gt', 'policy', 'ipcc_low', 'ipcc_high']
Countries:   ['UK', 'US', 'EU', 'China', 'Middle East', 'Australia', 'Canada', 'Indonesia', 'Thailand', 'Brazil']
Models:      ['Logistic', 'Gompertz']
Orders:      ['descend', 'ascend']
Checkpoints: [2050, 2100, 2179]
Horizon:     2030 – 2179 (150 yr)


/opt/anaconda3/envs/pygis/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load precomputed resource matrix (from Step 1)


In [2]:
resource_long = pd.read_csv(PRECOMPUTE_DIR / 'basin_period_resource_long.csv')
metadata_df = pd.read_csv(PRECOMPUTE_DIR / 'basin_metadata.csv')
print(f'Resource matrix: {len(resource_long):,} rows  '
      f'({resource_long["Region_no"].nunique()} basins × '
      f'{resource_long["Period_yr"].nunique()} periods)')


Resource matrix: 1,845 rows  (123 basins × 15 periods)


## Prepare growth-path inputs

Load yearly fixed-C deterministic central growth timeseries from Part 1 and convert Gt/yr → Mt/yr.


In [3]:
growth_raw = pd.read_csv(GROWTH_TIMESERIES)
print(f'Loaded {len(growth_raw):,} rows from growth timeseries')

required_growth_cols = {'Country', 'Scenario', 'Year', 'Model', 'rate_central'}
missing_growth_cols = required_growth_cols.difference(growth_raw.columns)
if missing_growth_cols:
    raise ValueError(f'Missing required V8 growth columns: {sorted(missing_growth_cols)}')


# Build screening input table
COUNTRY_TO_CASE = {c: c for c in SELECTED_CASES}
screening_input_all = pd.DataFrame({
    'Country': growth_raw['Country'],
    'Scenario': growth_raw['Scenario'],
    'Year': growth_raw['Year'].astype(int),
    'Model': growth_raw['Model'],
    'Rate_Mt_yr': growth_raw['rate_central'] * 1000.0,
})
# Keep only the active V8 study regions
screening_input_all = screening_input_all[
    screening_input_all['Country'].isin(SELECTED_CASES)
].sort_values(['Scenario', 'Country', 'Model', 'Year']).reset_index(drop=True)

print(f'Screening input: {len(screening_input_all):,} rows')
print(f'Scenarios with data: {sorted(screening_input_all["Scenario"].unique())}')

# Show availability matrix
avail = screening_input_all.groupby(['Scenario', 'Country', 'Model']).size().reset_index(name='n_years')
avail_pivot = avail.pivot_table(index='Country', columns='Scenario', values='n_years',
                                 aggfunc='count').fillna(0).astype(int)
print('\nData availability (country × scenario, count of model-year combos):')
print(avail_pivot.to_string())


Loaded 22,500 rows from growth timeseries
Screening input: 22,500 rows
Scenarios with data: ['growth10', 'ipcc_high', 'ipcc_low', 'maximum', 'minimum', 'policy', 'reference', 'us1gt']

Data availability (country × scenario, count of model-year combos):
Scenario     growth10  ipcc_high  ipcc_low  maximum  minimum  policy  reference  us1gt
Country                                                                               
Australia           2          2         2        2        2       2          2      2
Brazil              2          1         0        2        2       2          2      2
Canada              2          2         1        2        2       2          2      2
China               2          0         2        2        2       2          2      2
EU                  2          2         2        2        2       2          2      2
Indonesia           2          0         2        2        2       2          2      2
Middle East         2          2         2        2

## Run allocation for all combinations

Loop: `allocation_order × scenario × country × model`

Each allocation order writes to its own output tree:
- `descending/` for `allocation_order = 'descend'`
- `ascending/` for `allocation_order = 'ascend'`


In [4]:
SKIP_ALLOCATION = True   # Skip allocation; load cached screening_summary.csv from disk (figures-only re-run)

if SKIP_ALLOCATION:
    print('SKIP_ALLOCATION = True — loading existing screening_summary.csv files')
    summary_parts = []
    for alloc_order in ALLOCATION_ORDERS:
        order_label = ORDER_DIR_MAP[alloc_order]
        csv_path = OUTPUT_ROOT / order_label / 'final' / 'screening_summary.csv'
        if csv_path.exists():
            summary_parts.append(pd.read_csv(csv_path))
            print(f'  Loaded: {csv_path} ({len(summary_parts[-1])} rows)')
        else:
            print(f'  NOT FOUND: {csv_path}')
    if summary_parts:
        summary_df = pd.concat(summary_parts, ignore_index=True)
        print(f'Total: {len(summary_df)} runs ({(summary_df["status"] == "ok").sum()} ok)')
    else:
        raise FileNotFoundError('No screening_summary.csv found. Run allocation first.')
else:
        # Build rate/site pivot matrices (shared across all runs)
    periods = list(range(STORAGE_PERIOD_STEP_YR, INJ_DURATION_YR + 1, STORAGE_PERIOD_STEP_YR))
    rate_pivot = resource_long.pivot_table(
        index=['Region_no', 'Region_name'], columns='Period_yr',
        values='Region_Q_Mt_yr', aggfunc='first').reset_index()
    site_pivot = resource_long.pivot_table(
        index=['Region_no', 'Region_name'], columns='Period_yr',
        values='Number_of_Wells', aggfunc='first').reset_index()
    rate_pivot.columns = ['Region_no', 'Region_name'] + [
        f'Q [Mt/y] for t= {int(c)} y' for c in rate_pivot.columns[2:]]
    site_pivot.columns = ['Region_no', 'Region_name'] + [
        f'Q [Mt/y] for t= {int(c)} y' for c in site_pivot.columns[2:]]

    # Master summary collector
    all_run_summaries = []

    # Count total expected runs for progress bar
    total_runs = len(ALLOCATION_ORDERS) * len(SCENARIOS) * len(SELECTED_CASES) * len(MODELS)
    pbar = tqdm(total=total_runs, desc='Allocation runs', unit='run')

    for alloc_order in ALLOCATION_ORDERS:
        order_label = ORDER_DIR_MAP[alloc_order]
        order_root = OUTPUT_ROOT / order_label
        order_intermediate = order_root / 'intermediate'
        order_final = order_root / 'final'
        order_figures = order_root / 'figures'
        for d in [order_intermediate, order_final, order_figures]:
            d.mkdir(parents=True, exist_ok=True)

        for scenario in SCENARIOS:
            scenario_input = screening_input_all[screening_input_all['Scenario'] == scenario]

            for case in SELECTED_CASES:
                # Resolve basins
                region_nos, screening_basis = resolve_region_nos(
                    metadata_df, case, scope=SCREENING_SCOPE)

                # Prepare country cache
                if region_nos:
                    country_rate = rate_pivot[rate_pivot['Region_no'].isin(region_nos)].copy()
                    country_site = site_pivot[site_pivot['Region_no'].isin(region_nos)].copy()
                    country_dir = order_root / slugify(case)
                    country_dir.mkdir(parents=True, exist_ok=True)
                    rate_cache = country_dir / 'resource_rate_matrix_cache.csv'
                    site_cache = country_dir / 'site_number_matrix_cache.csv'
                    country_rate.to_csv(rate_cache, index=False)
                    country_site.to_csv(site_cache, index=False)

                for model in MODELS:
                    pbar.update(1)

                    # No matched basins for this study region
                    if not region_nos:
                        all_run_summaries.append({
                            'allocation_order': alloc_order,
                            'scenario': scenario, 'pathway': GROWTH_PATHWAY_LABEL,
                            'country': case, 'model': model,
                            'status': 'no_basin_identified',
                            'basin_match_basis': screening_basis,
                            'basin_count': 0,
                            'peak_demand_mt_yr': None, 'allocated_peak_mt_yr': 0.0,
                        })
                        continue

                    # No growth path for this combo
                    case_model_input = scenario_input[
                        (scenario_input['Country'] == case) &
                        (scenario_input['Model'] == model)
                    ].sort_values('Year')
                    if case_model_input.empty:
                        all_run_summaries.append({
                            'allocation_order': alloc_order,
                            'scenario': scenario, 'pathway': GROWTH_PATHWAY_LABEL,
                            'country': case, 'model': model,
                            'status': 'no_growth_path',
                            'basin_match_basis': screening_basis,
                            'basin_count': len(country_rate),
                            'peak_demand_mt_yr': None, 'allocated_peak_mt_yr': 0.0,
                        })
                        continue

                    # Run allocation
                    model_dir = order_root / slugify(case) / f'{scenario}_{model.lower()}'
                    model_dir.mkdir(parents=True, exist_ok=True)
                    gc_path = model_dir / 'growth_curve.csv'
                    case_model_input[['Year', 'Rate_Mt_yr']].rename(
                        columns={'Year': 'year', 'Rate_Mt_yr': 'total_rate'}
                    ).to_csv(gc_path, index=False)

                    try:
                        outputs = run_allocation_workflow(
                            AllocationConfig(
                                data_path=BASIN_FILE,
                                output_dir=model_dir,
                                nr_region=len(country_rate),
                                correction='off',
                                dist_min_km=2.0, dist_max_km='auto',
                                nr_dist=NR_DIST, nr_well_max='auto', rw_m=0.2,
                                max_q_mt_per_year=MAX_Q_MT_YR,
                                min_q_mt_per_year=MIN_Q_MT_YR,
                                inj_duration_yr=INJ_DURATION_YR,
                                allocation_duration_yr=ALLOCATION_DURATION_YR,
                                allocation_order=alloc_order,
                                storage_resource_calculation='savedfile',
                                storage_period_step_yr=STORAGE_PERIOD_STEP_YR,
                                growth_curve_path=gc_path,
                                resource_rate_cache_path=rate_cache,
                                site_no_cache_path=site_cache,
                            )
                        )
                        at = outputs['assignment_table']
                        peak_alloc = float(at['Step rate cumulative [Mt/y]'].max()) if not at.empty else 0.0
                        peak_demand = float(case_model_input['Rate_Mt_yr'].max())
                        status = 'ok'
                    except Exception as exc:
                        peak_alloc = 0.0
                        peak_demand = float(case_model_input['Rate_Mt_yr'].max())
                        status = f'allocation_error: {exc}'

                    all_run_summaries.append({
                        'allocation_order': alloc_order,
                        'scenario': scenario, 'pathway': GROWTH_PATHWAY_LABEL,
                        'country': case, 'model': model,
                        'status': status,
                        'basin_match_basis': screening_basis,
                        'basin_count': len(country_rate),
                        'peak_demand_mt_yr': peak_demand,
                        'allocated_peak_mt_yr': peak_alloc,
                    })

    pbar.close()

    # Save run-level summary per allocation order
    summary_df = pd.DataFrame(all_run_summaries)
    for alloc_order in ALLOCATION_ORDERS:
        order_label = ORDER_DIR_MAP[alloc_order]
        sub = summary_df[summary_df['allocation_order'] == alloc_order]
        sub.to_csv(OUTPUT_ROOT / order_label / 'final' / 'screening_summary.csv', index=False)

    ok_count = (summary_df['status'] == 'ok').sum()
    skip_count = len(summary_df) - ok_count
    print(f'\nDone: {ok_count} successful runs, {skip_count} skipped')
    print(f'Statuses: {summary_df["status"].value_counts().to_dict()}')


SKIP_ALLOCATION = True — loading existing screening_summary.csv files
  Loaded: /Users/xg320/Desktop/IC-Sam-works/paper/Iman-2026/02_co2block_screening/output/step2_screening/descending/final/screening_summary.csv (160 rows)
  Loaded: /Users/xg320/Desktop/IC-Sam-works/paper/Iman-2026/02_co2block_screening/output/step2_screening/ascending/final/screening_summary.csv (160 rows)
Total: 320 runs (300 ok)


## Post-processing: reconstruct screened paths & multi-checkpoint metrics

For each successful allocation run, reconstruct:
- Annual and cumulative screened paths (2030–2179)
- Metrics at checkpoints: 2050, 2100, 2179


In [5]:
def year_active_mask(years, start, end):
    return (years >= start) & (years <= end)

def compute_checkpoint_metrics(screened_df, basin_alloc_df, checkpoint_year):
    """Compute all metrics at a single checkpoint year."""
    mask = screened_df['year'] <= checkpoint_year
    sc = screened_df[mask]
    if sc.empty:
        return {}

    # Rate at checkpoint
    cp_row = sc[sc['year'] == checkpoint_year]
    if cp_row.empty:
        # Use last available year
        cp_row = sc.iloc[[-1]]

    annual_demand = float(cp_row['raw_rate_mt_yr'].iloc[0])
    annual_screened = float(cp_row['screened_rate_mt_yr'].iloc[0])
    annual_gap = float(cp_row['gap_rate_mt_yr'].iloc[0])
    r_annual = annual_screened / annual_demand if annual_demand > 0 else 0.0

    # Cumulative up to checkpoint
    cum_demand = float(sc['raw_rate_mt_yr'].sum() / 1000.0)
    cum_screened = float(sc['screened_rate_mt_yr'].sum() / 1000.0)
    cum_gap = cum_demand - cum_screened
    r_cumulative = cum_screened / cum_demand if cum_demand > 0 else 0.0

    # Pass/fail
    annual_pass = r_annual >= 1.0 - 1e-6
    cumulative_pass = r_cumulative >= 1.0 - 1e-6
    overall_pass = annual_pass and cumulative_pass

    # Shortfall diagnostics within [2030, checkpoint]
    shortfalls = sc[sc['gap_rate_mt_yr'] > 1e-9]
    first_shortfall_year = int(shortfalls['year'].iloc[0]) if not shortfalls.empty else None
    years_of_shortfall = len(shortfalls)
    max_gap_rate = float(sc['gap_rate_mt_yr'].max())
    max_gap_year = int(sc.loc[sc['gap_rate_mt_yr'].idxmax(), 'year']) if max_gap_rate > 1e-9 else None
    total_unmet_gt = float(sc['gap_rate_mt_yr'].clip(lower=0).sum() / 1000.0)

    # Portfolio structure at checkpoint
    active_basins = 0
    total_wells = 0
    n_eff = 0.0
    largest_share = 0.0
    dominant_basin = None
    total_basins = 0

    if not basin_alloc_df.empty:
        total_basins = basin_alloc_df['basin'].nunique()
        at_cp = basin_alloc_df[basin_alloc_df['year'] == checkpoint_year]
        if at_cp.empty:
            # Use last year with data <= checkpoint
            valid_years = basin_alloc_df[basin_alloc_df['year'] <= checkpoint_year]['year']
            if not valid_years.empty:
                at_cp = basin_alloc_df[basin_alloc_df['year'] == valid_years.max()]

        if not at_cp.empty:
            shares = at_cp.groupby('basin')['delivered_rate_mt_yr'].sum()
            active_basins = int((shares > 0).sum())
            total_wells = int(at_cp['wells_used'].sum())
            total = shares.sum()
            if total > 0:
                s = shares / total
                n_eff = float(1.0 / (s ** 2).sum())
                largest_share = float(s.max())
                dominant_basin = s.idxmax()

    return {
        'checkpoint': checkpoint_year,
        'annual_demand_mt_yr': round(annual_demand, 4),
        'annual_screened_mt_yr': round(annual_screened, 4),
        'annual_gap_mt_yr': round(annual_gap, 4),
        'r_annual': round(r_annual, 4),
        'cumulative_demand_gt': round(cum_demand, 4),
        'cumulative_screened_gt': round(cum_screened, 4),
        'cumulative_gap_gt': round(cum_gap, 4),
        'r_cumulative': round(r_cumulative, 4),
        'annual_pass': annual_pass,
        'cumulative_pass': cumulative_pass,
        'overall_pass': overall_pass,
        'first_shortfall_year': first_shortfall_year,
        'years_of_shortfall': years_of_shortfall,
        'max_gap_rate_mt_yr': round(max_gap_rate, 4),
        'max_gap_year': max_gap_year,
        'total_unmet_volume_gt': round(total_unmet_gt, 4),
        'active_basins': active_basins,
        'total_basins': total_basins,
        'active_basin_fraction': round(active_basins / total_basins, 4) if total_basins > 0 else 0.0,
        'total_wells': total_wells,
        'n_eff': round(n_eff, 2),
        'largest_basin_share': round(largest_share, 4),
        'dominant_basin_name': dominant_basin,
    }


def reconstruct_screened_paths(country, scenario, model, alloc_order,
                                screening_input_df, model_dir):
    """Reconstruct full screened path and compute all checkpoint metrics."""
    cm = screening_input_df[
        (screening_input_df['Country'] == country) &
        (screening_input_df['Scenario'] == scenario) &
        (screening_input_df['Model'] == model)
    ].sort_values('Year').reset_index(drop=True)

    years = cm['Year'].to_numpy(dtype=int)
    raw_rate = cm['Rate_Mt_yr'].to_numpy(dtype=float)

    # Limit to allocation window
    mask_window = years <= LAST_ALLOC_YEAR
    years_w = years[mask_window]
    raw_w = raw_rate[mask_window]

    assignment_path = model_dir / 'Resource_assignment_python.xlsx'

    # Build basin allocation year-by-year
    basin_rows = []
    if assignment_path.exists():
        assignment = pd.read_excel(assignment_path)
        for _, row in assignment.iterrows():
            basin = str(row['Region name'])
            delivered_rate = float(row['Step rate increment [Mt/y]'])
            wells = float(row['No_sites'])
            start_y = float(row['Start [y]'])
            end_y = float(row['End [y]'])
            step = int(row['Step'])
            duration = float(row['Duration [y]'])
            region_no = int(row['Region no'])
            for yr in years_w[year_active_mask(years_w.astype(float), start_y, end_y)]:
                basin_rows.append({
                    'country': country, 'scenario': scenario, 'model': model,
                    'allocation_order': alloc_order,
                    'year': int(yr), 'basin': basin, 'region_no': region_no,
                    'allocated_rate_mt_yr': delivered_rate,
                    'delivered_rate_mt_yr': delivered_rate,
                    'wells_used': wells, 'step': step,
                    'duration': duration, 'start_year': start_y, 'end_year': end_y,
                })

    basin_alloc = pd.DataFrame(basin_rows)

    # Proportional clipping: delivered <= raw demand per year
    if not basin_alloc.empty:
        basin_alloc['delivered_rate_mt_yr'] = 0.0
        year_to_raw = dict(zip(years_w.tolist(), raw_w.tolist()))
        for yr in years_w:
            ym = basin_alloc['year'] == yr
            if not ym.any():
                continue
            remaining = float(year_to_raw[int(yr)])
            block = basin_alloc.loc[ym].sort_values(['step', 'start_year', 'basin'])
            for idx in block.index:
                delivered = min(float(basin_alloc.at[idx, 'allocated_rate_mt_yr']),
                               max(remaining, 0.0))
                basin_alloc.at[idx, 'delivered_rate_mt_yr'] = delivered
                remaining -= delivered

    # Annual screened path
    if not basin_alloc.empty:
        annual_total = basin_alloc.groupby('year', as_index=False)['delivered_rate_mt_yr'].sum()
    else:
        annual_total = pd.DataFrame({'year': years_w, 'delivered_rate_mt_yr': 0.0})
    annual_total = pd.DataFrame({'year': years_w}).merge(
        annual_total, on='year', how='left').fillna({'delivered_rate_mt_yr': 0.0})
    screened_rate = annual_total['delivered_rate_mt_yr'].values

    screened = pd.DataFrame({
        'country': country, 'scenario': scenario, 'pathway': GROWTH_PATHWAY_LABEL,
        'model': model, 'allocation_order': alloc_order,
        'year': years_w,
        'raw_rate_mt_yr': raw_w,
        'screened_rate_mt_yr': screened_rate,
    })
    screened['gap_rate_mt_yr'] = screened['raw_rate_mt_yr'] - screened['screened_rate_mt_yr']
    screened['raw_cumulative_gt'] = screened['raw_rate_mt_yr'].cumsum() / 1000.0
    screened['screened_cumulative_gt'] = screened['screened_rate_mt_yr'].cumsum() / 1000.0
    screened['gap_cumulative_gt'] = screened['raw_cumulative_gt'] - screened['screened_cumulative_gt']

    # Checkpoint metrics
    checkpoint_metrics = []
    for cp in CHECKPOINTS:
        m = compute_checkpoint_metrics(screened, basin_alloc, cp)
        if m:
            m.update({
                'country': country, 'scenario': scenario,
                'pathway': GROWTH_PATHWAY_LABEL, 'model': model,
                'allocation_order': alloc_order,
            })
            checkpoint_metrics.append(m)

    # Basin-level metrics at each checkpoint
    basin_metrics = []
    if not basin_alloc.empty:
        # Get max feasible from Step 1
        resource_lookup = resource_long.set_index(['Region_no', 'Period_yr'])['Region_Q_Mt_yr']
        for cp in CHECKPOINTS:
            at_cp = basin_alloc[basin_alloc['year'] == cp]
            if at_cp.empty:
                valid_y = basin_alloc[basin_alloc['year'] <= cp]['year']
                if not valid_y.empty:
                    at_cp = basin_alloc[basin_alloc['year'] == valid_y.max()]

            total_annual = at_cp['delivered_rate_mt_yr'].sum() if not at_cp.empty else 0
            cum_up_to = basin_alloc[basin_alloc['year'] <= cp]

            for basin_name in basin_alloc['basin'].unique():
                b_cp = at_cp[at_cp['basin'] == basin_name]
                b_cum = cum_up_to[cum_up_to['basin'] == basin_name]

                alloc_rate = float(b_cp['delivered_rate_mt_yr'].sum()) if not b_cp.empty else 0.0
                cum_delivered = float(b_cum['delivered_rate_mt_yr'].sum() / 1000.0)
                wells = int(b_cp['wells_used'].iloc[0]) if not b_cp.empty else 0
                step_rank = int(b_cp['step'].iloc[0]) if not b_cp.empty else 0
                duration = float(b_cp['duration'].iloc[0]) if not b_cp.empty else 0
                region_no = int(b_cp['region_no'].iloc[0]) if not b_cp.empty and 'region_no' in b_cp.columns else 0

                # Max feasible from Step 1 at assigned period
                max_feas = 0.0
                if region_no > 0 and duration > 0:
                    period_key = int(duration)
                    try:
                        max_feas = float(resource_lookup.get((region_no, period_key), 0.0))
                    except:
                        pass

                util = alloc_rate / max_feas if max_feas > 0 else 0.0

                # Country cumulative delivered to this checkpoint
                country_cum_to_cp = float(cum_up_to['delivered_rate_mt_yr'].sum() / 1000.0)

                basin_metrics.append({
                    'country': country, 'scenario': scenario, 'model': model,
                    'allocation_order': alloc_order, 'checkpoint': cp,
                    'basin': basin_name, 'region_no': region_no,
                    'allocated_rate_mt_yr': round(alloc_rate, 4),
                    'max_feasible_rate_mt_yr': round(max_feas, 4),
                    'utilisation_ratio': round(util, 4),
                    'delivered_cumulative_gt': round(cum_delivered, 6),
                    'share_of_country_annual': round(alloc_rate / total_annual, 4) if total_annual > 0 else 0.0,
                    'share_of_country_cumulative': round(cum_delivered / country_cum_to_cp, 4) if country_cum_to_cp > 0 else 0.0,
                    'wells_used': wells,
                    'step_rank': step_rank,
                    'assigned_period_yr': int(duration),
                })

    return basin_alloc, screened, checkpoint_metrics, basin_metrics


# Run post-processing for all successful allocation runs
all_basin_alloc = []
all_screened = []
all_checkpoint_metrics = []
all_basin_metrics = []

ok_runs = summary_df[summary_df['status'] == 'ok']
print(f'Post-processing {len(ok_runs)} successful runs...')

for _, run in tqdm(ok_runs.iterrows(), total=len(ok_runs), desc='Post-processing', unit='run'):
    alloc_order = run['allocation_order']
    scenario = run['scenario']
    country = run['country']
    model = run['model']
    order_label = ORDER_DIR_MAP[alloc_order]
    model_dir = OUTPUT_ROOT / order_label / slugify(country) / f'{scenario}_{model.lower()}'

    ba, sc, cp_metrics, b_metrics = reconstruct_screened_paths(
        country, scenario, model, alloc_order,
        screening_input_all, model_dir)

    all_basin_alloc.append(ba)
    all_screened.append(sc)
    all_checkpoint_metrics.extend(cp_metrics)
    all_basin_metrics.extend(b_metrics)

# Save outputs per allocation order
if all_screened:
    screened_all = pd.concat(all_screened, ignore_index=True)
if all_basin_alloc:
    ba_all = pd.concat([b for b in all_basin_alloc if not b.empty], ignore_index=True)
cp_all = pd.DataFrame(all_checkpoint_metrics)
bm_all = pd.DataFrame(all_basin_metrics)

for alloc_order in ALLOCATION_ORDERS:
    order_label = ORDER_DIR_MAP[alloc_order]
    final_dir = OUTPUT_ROOT / order_label / 'final'
    inter_dir = OUTPUT_ROOT / order_label / 'intermediate'

    if all_screened:
        screened_all[screened_all['allocation_order'] == alloc_order].to_csv(
            final_dir / 'case_model_screened_paths.csv', index=False)
    # NOTE: legacy intermediate/case_model_basin_allocation.csv is no longer
    # persisted under Path D-full; the same information is now in
    # case_model_basin_metrics.csv (final/) and case_model_basin_exhaustion.csv.
    if not cp_all.empty:
        cp_all[cp_all['allocation_order'] == alloc_order].to_csv(
            final_dir / 'case_model_portfolio_metrics.csv', index=False)
    if not bm_all.empty:
        bm_all[bm_all['allocation_order'] == alloc_order].to_csv(
            final_dir / 'case_model_basin_metrics.csv', index=False)

print(f'\nScreened paths: {len(screened_all):,} rows')
print(f'Checkpoint metrics: {len(cp_all):,} rows')
print(f'Basin metrics: {len(bm_all):,} rows')


# NEW (Path D-full): whole-period diagnostics + basin exhaustion
# Whole-period shortfall: a case passes if there is no year between 2030
# and LAST_ALLOC_YEAR where the screened rate drops below the demand
# rate. Five secondary diagnostics characterise the shortfall window.
GAP_TOL_MT = 0.01   # ignore numerical noise

def _compute_whole_period_diagnostics(scrn):
    """Compute whole-horizon shortfall summary for a single case sub-df.
    Input: scrn (DataFrame with columns 'year', 'raw_rate_mt_yr',
    'screened_rate_mt_yr') already filtered to a single (country, scen,
    model, order) and sorted by year."""
    if scrn.empty:
        return None
    years    = scrn['year'].to_numpy(dtype=int)
    raw      = scrn['raw_rate_mt_yr'].to_numpy(dtype=float)
    screened = scrn['screened_rate_mt_yr'].to_numpy(dtype=float)
    gap      = np.maximum(raw - screened, 0.0)
    is_short = gap > GAP_TOL_MT
    cum_demand_gt   = float(raw.sum() / 1000.0)
    cum_screened_gt = float(screened.sum() / 1000.0)
    if not is_short.any():
        return {
            'whole_period_pass': True,
            'years_of_shortfall': 0,
            'first_shortfall_year': None,
            'last_shortfall_year': None,
            'shortfall_window_yr': 0,
            'max_gap_rate_mt_yr': 0.0,
            'max_gap_year': None,
            'total_unmet_volume_gt': 0.0,
            'cumulative_demand_full_gt': round(cum_demand_gt, 4),
            'cumulative_screened_full_gt': round(cum_screened_gt, 4),
            'overall_r_cumulative': round(cum_screened_gt / cum_demand_gt, 4)
                                     if cum_demand_gt > 0 else float('nan'),
        }
    short_years = years[is_short]
    max_gap_idx = int(np.argmax(gap))
    return {
        'whole_period_pass': False,
        'years_of_shortfall': int(is_short.sum()),
        'first_shortfall_year': int(short_years[0]),
        'last_shortfall_year': int(short_years[-1]),
        'shortfall_window_yr': int(short_years[-1] - short_years[0] + 1),
        'max_gap_rate_mt_yr': round(float(gap.max()), 4),
        'max_gap_year': int(years[max_gap_idx]),
        'total_unmet_volume_gt': round(float(gap.sum() / 1000.0), 4),
        'cumulative_demand_full_gt': round(cum_demand_gt, 4),
        'cumulative_screened_full_gt': round(cum_screened_gt, 4),
        'overall_r_cumulative': round(cum_screened_gt / cum_demand_gt, 4)
                                 if cum_demand_gt > 0 else float('nan'),
    }


# Iterate over every successful case and compute the whole-period
# diagnostic (the only per-case basin-level summary we need)
sf_rows = []
print('\nComputing whole-period diagnostics ...')
for _, run in tqdm(ok_runs.iterrows(), total=len(ok_runs),
                    desc='Whole-period diag', unit='run'):
    co, sc_, md, ao = run['country'], run['scenario'], run['model'], run['allocation_order']
    scrn = screened_all[
        (screened_all['country'] == co) &
        (screened_all['scenario'] == sc_) &
        (screened_all['model'] == md) &
        (screened_all['allocation_order'] == ao)
    ].sort_values('year')
    diag = _compute_whole_period_diagnostics(scrn)
    if diag is not None:
        sf_rows.append({'country': co, 'scenario': sc_, 'model': md,
                        'allocation_order': ao, **diag})

sf_df = pd.DataFrame(sf_rows)

for alloc_order in ALLOCATION_ORDERS:
    order_label = ORDER_DIR_MAP[alloc_order]
    final_dir = OUTPUT_ROOT / order_label / 'final'
    final_dir.mkdir(parents=True, exist_ok=True)
    if not sf_df.empty:
        sf_df[sf_df['allocation_order'] == alloc_order].to_csv(
            final_dir / 'case_model_shortfall_windows.csv', index=False)

print(f'\nWhole-period diagnostics: {len(sf_df)} rows')
print(f'  whole_period_pass: '
      f'{int(sf_df["whole_period_pass"].sum())} pass / '
      f'{int((~sf_df["whole_period_pass"]).sum())} fail')


Post-processing 300 successful runs...


Post-processing:   0%|          | 0/300 [00:00<?, ?run/s]

Post-processing:   0%|          | 1/300 [00:00<00:34,  8.66run/s]

Post-processing:   1%|          | 3/300 [00:00<00:20, 14.23run/s]

Post-processing:   2%|▏         | 6/300 [00:00<00:16, 17.86run/s]

...

Whole-period diag:  75%|███████▍  | 224/300 [00:00<00:00, 276.79run/s]

Whole-period diag:  84%|████████▍ | 252/300 [00:00<00:00, 276.01run/s]

Whole-period diag:  93%|█████████▎| 280/300 [00:01<00:00, 275.47run/s]

Whole-period diag: 100%|██████████| 300/300 [00:01<00:00, 276.02run/s]


Whole-period diagnostics: 300 rows
  whole_period_pass: 222 pass / 78 fail


## Figure 3: Whole-period feasibility scorecards (asc / desc)

A single 2×2 figure showing the whole-period verdict under both
allocation orderings:

* Top row: `ascending` (smallest basin first)
* Bottom row: `descending` (largest basin first)
* Columns: `Logistic` | `Gompertz`

Each cell encodes one `(country, scenario, model, ordering)` verdict
using the severity-tier colour scheme (green / amber / orange / red)
with the cell label `Xy / Y Gt` for failing cases or a pass mark for
passing cases. A top banner reports the order-sensitivity headline
(pass-flips = 0/150, plus the small number of cells that differ in
shortfall years).

Saved at `output/step2_screening/fig3_dashboard.{pdf,png}` (single
file; no per-ordering duplicate needed since both orderings live in
one figure).

The Q1-vs-Q2-vs-Q3 split is strict: Fig 3 covers Q1 only. The
ordering comparison (Q3) lives in its own figure, see the next
section.

In [6]:
# 
# Figure 3: Whole-period feasibility scorecards under both
# allocation orderings (Path D-full).
#
# Layout: 2 rows (ascending / descending) × 2 cols (Logistic / Gompertz).
# Each panel is the same scorecard heatmap (severity tiers). Top
# banner reports the order-sensitivity headline.
#
# This figure is pure Q1 (does the basin portfolio match the growth
# curve?). Q3 (ordering comparison on severity + wells burden) is in
# `fig_ordering_comparison.{pdf,png}`, produced in the next cell.
# 

plt.rcParams.update({
    'font.size': 12, 'axes.titlesize': 13, 'axes.labelsize': 12,
    'xtick.labelsize': 10, 'ytick.labelsize': 10, 'legend.fontsize': 9,
    'axes.spines.top': False, 'axes.spines.right': False,
})

_CPAL = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2', '#59a14f',
         '#edc948', '#b07aa1', '#ff9da7', '#9c755f', '#bab0ac']  # 10 colours for the v8 10-region pool
COUNTRY_COLORS = {c: _CPAL[i % len(_CPAL)] for i, c in enumerate(SELECTED_CASES)}
MODEL_MARKERS  = {'Logistic': 'o', 'Gompertz': '^'}

from matplotlib.patches import Patch, Rectangle
from matplotlib.gridspec import GridSpecFromSubplotSpec


def _severity_tier(years_short, unmet_gt):
    if years_short is None or unmet_gt is None:
        return 0, '#1a9850', 'white'
    if years_short == 0 and unmet_gt < 0.01:
        return 0, '#1a9850', 'white'
    if years_short <= 10 and unmet_gt < 5.0:
        return 1, '#fee08b', '#333'
    if years_short <= 25 or unmet_gt < 50.0:
        return 2, '#f4a582', 'white'
    return 3, '#d73027', 'white'


def _scorecard_cell_text(years_short, unmet_gt):
    if years_short == 0:
        return '✓ pass'
    return f'{int(years_short)}y\n{unmet_gt:.0f} Gt'


def _draw_scorecard_panel(ax, sf_sub, model_name, countries, scenarios,
                            row_label):
    """One scorecard heatmap for (ordering, model). row_label is
    either 'ascending' or 'descending'; used in the panel title."""
    n_rows = len(countries); n_cols = len(scenarios)
    for ri, country in enumerate(countries):
        for ci, scenario in enumerate(scenarios):
            row = sf_sub[(sf_sub['country'] == country) &
                         (sf_sub['scenario'] == scenario) &
                         (sf_sub['model'] == model_name)]
            if row.empty:
                color, text, tc = '#f5f5f5', 'N/A', '#aaa'
            else:
                ys = int(row['years_of_shortfall'].iloc[0])
                uv = float(row['total_unmet_volume_gt'].iloc[0])
                _, color, tc = _severity_tier(ys, uv)
                text = _scorecard_cell_text(ys, uv)
            ax.add_patch(Rectangle(
                (ci - 0.48, ri - 0.42), 0.96, 0.84,
                facecolor=color, edgecolor='white', lw=1.5))
            ax.text(ci, ri, text, ha='center', va='center',
                    fontsize=9.5, fontweight='bold', color=tc)
    ax.set_xlim(-0.5, n_cols - 0.5)
    ax.set_ylim(n_rows - 0.5, -0.5)
    ax.set_xticks(range(n_cols))
    ax.set_xticklabels(scenarios, rotation=45, ha='right', fontsize=9.5)
    ax.set_yticks(range(n_rows))
    ax.set_yticklabels(countries, fontsize=10.5)
    ax.set_title(
        f'{row_label} · {model_name}',
        fontsize=12, fontweight='bold', pad=8, loc='left')
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)


def _build_deltas_table(sf_df, cp_all):
    """Per-case desc − asc delta table used by both Fig 3 banner and
    the new ordering-comparison figure (next cell)."""
    KEY = ['country', 'scenario', 'model']
    CHECKPOINTS = [2050, 2100, 2179]
    sf_a = sf_df[sf_df['allocation_order'] == 'ascend'][
        KEY + ['years_of_shortfall', 'total_unmet_volume_gt',
               'whole_period_pass']].copy()
    sf_d = sf_df[sf_df['allocation_order'] == 'descend'][
        KEY + ['years_of_shortfall', 'total_unmet_volume_gt',
               'whole_period_pass']].copy()
    sm = sf_a.merge(sf_d, on=KEY, suffixes=('_asc', '_desc'))
    sm['d_years_of_shortfall'] = (
        sm['years_of_shortfall_desc'] - sm['years_of_shortfall_asc'])
    sm['d_total_unmet_gt'] = (
        sm['total_unmet_volume_gt_desc'] -
        sm['total_unmet_volume_gt_asc'])
    sm['pass_flip'] = (sm['whole_period_pass_asc'] !=
                        sm['whole_period_pass_desc'])
    deltas = sm.copy()
    for cp in CHECKPOINTS:
        cp_sub = cp_all[cp_all['checkpoint'] == cp][
            KEY + ['allocation_order', 'total_wells', 'n_eff',
                   'active_basins']]
        cp_a = cp_sub[cp_sub['allocation_order'] == 'ascend'].drop(
            columns=['allocation_order'])
        cp_d = cp_sub[cp_sub['allocation_order'] == 'descend'].drop(
            columns=['allocation_order'])
        cm = cp_a.merge(cp_d, on=KEY,
                         suffixes=(f'_asc_{cp}', f'_desc_{cp}'))
        cm[f'd_total_wells_{cp}']   = (
            cm[f'total_wells_desc_{cp}']   - cm[f'total_wells_asc_{cp}'])
        cm[f'd_n_eff_{cp}']         = (
            cm[f'n_eff_desc_{cp}']         - cm[f'n_eff_asc_{cp}'])
        cm[f'd_active_basins_{cp}'] = (
            cm[f'active_basins_desc_{cp}'] - cm[f'active_basins_asc_{cp}'])
        deltas = deltas.merge(cm, on=KEY)
    return deltas


def _tier_index(years_short, unmet_gt):
    """Return the 0/1/2/3 tier index used by `_severity_tier`."""
    if years_short == 0 and unmet_gt < 0.01:
        return 0
    if years_short <= 10 and unmet_gt < 5.0:
        return 1
    if years_short <= 25 or unmet_gt < 50.0:
        return 2
    return 3


def make_ordered_scorecard_figure(sf_df, cp_all, scenarios, countries,
                                    models, output_path):
    """Render the 2x2 scorecard Fig 3 plus its order-sensitivity banner.

    Top row = ascending, bottom row = descending; columns = models.
    """
    deltas = _build_deltas_table(sf_df, cp_all)
    n_total    = len(deltas)
    n_flips    = int(deltas['pass_flip'].sum())
    n_diff_yrs = int((deltas['d_years_of_shortfall'] != 0).sum())
    n_diff_um  = int((deltas['d_total_unmet_gt'].abs() > 0.1).sum())
    # Tier-flip count
    deltas['tier_asc'] = [
        _tier_index(int(y), float(u))
        for y, u in zip(deltas['years_of_shortfall_asc'],
                         deltas['total_unmet_volume_gt_asc'])]
    deltas['tier_desc'] = [
        _tier_index(int(y), float(u))
        for y, u in zip(deltas['years_of_shortfall_desc'],
                         deltas['total_unmet_volume_gt_desc'])]
    n_tier_flips = int((deltas['tier_asc'] != deltas['tier_desc']).sum())

    fig = plt.figure(figsize=(20, 18))
    outer = fig.add_gridspec(
        2, 2,
        left=0.06, right=0.985, top=0.86, bottom=0.06,
        hspace=0.42, wspace=0.18)

    ROW_LABELS = ['ascending (smallest basin first)',
                   'descending (largest basin first)']
    ORDERS = ['ascend', 'descend']

    for ri, (row_label, order) in enumerate(zip(ROW_LABELS, ORDERS)):
        sf_sub = sf_df[sf_df['allocation_order'] == order].copy()
        for ci, mdl in enumerate(models):
            ax = fig.add_subplot(outer[ri, ci])
            _draw_scorecard_panel(
                ax, sf_sub, mdl, countries, scenarios, row_label)

    # Suptitle
    fig.suptitle(
        'Figure 3 — Whole-period feasibility scorecards (Path D-full)\n'
        'Top row: ascending · Bottom row: descending · '
        'Columns: Logistic, Gompertz',
        fontsize=15, fontweight='bold', y=0.975)

    # Headline banner (order-sensitivity summary)
    banner = (
        f'Order-sensitivity headline (descend − ascend, n = {n_total}): '
        f'pass-flips = {n_flips}/{n_total} · '
        f'cells differing on years = {n_diff_yrs}/{n_total} · '
        f'cells differing on |Δ unmet| > 0.1 Gt = {n_diff_um}/{n_total} · '
        f'severity-tier flips = {n_tier_flips}/{n_total}.\n'
        'Full Q3 evidence: fig_ordering_comparison.{pdf,png} '
        '+ extended_data/ordering_sensitivity_deltas.csv.'
    )
    fig.text(0.5, 0.915, banner, ha='center', va='top',
              fontsize=10, style='italic', color='#444')

    # Severity legend (shared, placed bottom-left of figure)
    legend_text = (
        'Severity tier: green ✓ pass · amber minor (≤10y, <5 Gt) · '
        'orange moderate · red severe (>25y or ≥50 Gt unmet)'
    )
    fig.text(0.5, 0.025, legend_text, ha='center', va='bottom',
              fontsize=9.5, color='#444')

    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(str(output_path) + '.pdf', dpi=180, bbox_inches='tight')
    fig.savefig(str(output_path) + '.png', dpi=180, bbox_inches='tight')
    plt.close(fig)
    print(f'  wrote {output_path.relative_to(OUTPUT_ROOT.parent)}.{{pdf,png}}')

    # Persist the deltas CSV (still the audit-trail file)
    deltas_dir = OUTPUT_ROOT / 'extended_data'
    deltas_dir.mkdir(parents=True, exist_ok=True)
    deltas.drop(columns=['tier_asc', 'tier_desc']).to_csv(
        deltas_dir / 'ordering_sensitivity_deltas.csv', index=False)
    print(f'  wrote {(deltas_dir / "ordering_sensitivity_deltas.csv").relative_to(OUTPUT_ROOT.parent)}')


print('Generating Fig 3 (2×2 scorecards) ...')
make_ordered_scorecard_figure(
    sf_df=sf_df,
    cp_all=cp_all,
    scenarios=SCENARIOS,
    countries=SELECTED_CASES,
    models=MODELS,
    output_path=OUTPUT_ROOT / 'fig3_dashboard',
)
print('\nFig 3 rendered (pure Q1; Q3 lives in the next figure).')


Generating Fig 3 (2×2 scorecards) ...


  wrote step2_screening/fig3_dashboard.{pdf,png}
  wrote step2_screening/extended_data/ordering_sensitivity_deltas.csv

Fig 3 rendered (pure Q1; Q3 lives in the next figure).


## Ordering comparison: severity vs CAPEX-related burden (Q3)

A dedicated figure that answers Q3: does the choice between ascending
and descending ordering change the feasibility verdict, modestly
change the failure severity, or strongly change the operational
well fleet?

The figure has two panels:

* Panel (a): per-case `Δ total_unmet_gt` (descending − ascending),
  ranked by absolute value. One dot per case, country colour, model
  marker. The headline `0/150 pass-flips` plus the shortfall-year
  difference count is annotated at the top of the panel.
* Panel (b): distribution of `|Δ total_wells|` (descending −
  ascending) at the three checkpoints 2050, 2100, 2179. Box-and-strip
  plot showing the median, IQR, and individual case outliers.

Saved at `output/step2_screening/fig_ordering_comparison.{pdf,png}`.
The underlying per-case table is
`output/step2_screening/extended_data/ordering_sensitivity_deltas.csv`.

Headline result: feasibility is order-invariant (0/150 pass-flips);
failure severity moves modestly (19/150 differ on years, 38/150 on
|Δ unmet| > 0.1 Gt); the operational well fleet differs much more
systematically (median |Δ| @ 2100 = 202, max = 2 882).

In [7]:
# 
# Ordering comparison figure, Q3
#
# Two-panel figure addressing the ordering-sensitivity question:
#
#   Panel (a): per-case Δ total_unmet_gt, sorted by |Δ|.
#              Answers "how much does ordering change failure
#              severity?"
#   Panel (b): |Δ total_wells| distribution at 2050/2100/2179.
#              Answers "how much does ordering change the operational
#              well-fleet burden?"
#
# Both panels share country colouring and model markers. The result
# is the complete Q3 evidence in one figure.
# 

def _annotate_top_outliers(ax, df, y_col, n=5, x_col=None):
    """Label the n largest |y_col| cases with leader lines."""
    if df.empty:
        return
    top = df.assign(_a=df[y_col].abs()).nlargest(n, '_a').reset_index(drop=True)
    offsets = [(8, 14), (8, -14), (-8, -14), (-8, 14),
                (8, 0), (-8, 0)]
    for i, (_, r) in enumerate(top.iterrows()):
        ox, oy = offsets[i % len(offsets)]
        ha = 'left' if ox > 0 else ('right' if ox < 0 else 'center')
        va = 'bottom' if oy > 0 else ('top' if oy < 0 else 'center')
        label = f"{r['country']} · {r['scenario']} · {r['model'][0]}"
        x = r[x_col] if x_col is not None else r.name
        ax.annotate(label, xy=(x, r[y_col]),
                     xytext=(ox, oy), textcoords='offset points',
                     fontsize=7.5, color='#333', ha=ha, va=va,
                     arrowprops=dict(arrowstyle='-', color='#999',
                                      lw=0.4, alpha=0.7))


def _panel_a_severity(ax, deltas):
    """Panel (a): per-case Δ unmet, ranked by |Δ|."""
    d = deltas.copy()
    d = d.sort_values('d_total_unmet_gt', key=abs,
                       ascending=True).reset_index(drop=True)
    d['rank'] = np.arange(len(d))
    for _, r in d.iterrows():
        ax.scatter(r['rank'], r['d_total_unmet_gt'],
                    s=45,
                    marker=MODEL_MARKERS[r['model']],
                    facecolors=COUNTRY_COLORS.get(r['country'], '#999'),
                    edgecolors='white', linewidth=0.4, alpha=0.85,
                    zorder=3)
    ax.axhline(0, color='#777', linestyle='--', lw=0.8, alpha=0.7)
    ax.set_xlabel('Case rank (sorted by |Δ total_unmet_gt|, ascending)',
                  fontsize=11)
    ax.set_ylabel('Δ total_unmet_gt = desc − asc  (Gt CO₂)', fontsize=11)
    ax.set_title(
        '(a) Severity difference per case',
        fontsize=12, fontweight='bold', pad=8, loc='left')
    ax.grid(True, linestyle=':', lw=0.6, alpha=0.4)
    # Label the top-5 outliers by |Δ unmet|
    _annotate_top_outliers(ax, d, 'd_total_unmet_gt', n=5, x_col='rank')


def _panel_b_wells(ax, deltas):
    """Panel (b): |Δ wells| distribution at 2050 / 2100 / 2179."""
    cps = [2050, 2100, 2179]
    # Build a long-form for the strip plot
    rows = []
    for cp in cps:
        for _, r in deltas.iterrows():
            rows.append({
                'checkpoint': cp,
                'abs_d_wells': abs(r[f'd_total_wells_{cp}']),
                'country': r['country'],
                'model': r['model'],
            })
    long = pd.DataFrame(rows)

    # Box plot
    box_data = [long[long['checkpoint'] == cp]['abs_d_wells'].values
                for cp in cps]
    bp = ax.boxplot(
        box_data, positions=range(len(cps)),
        widths=0.55, showfliers=False, patch_artist=True,
        medianprops=dict(color='#222', lw=1.4),
        boxprops=dict(facecolor='#e8e8e8', edgecolor='#666', lw=0.8),
        whiskerprops=dict(color='#666', lw=0.8),
        capprops=dict(color='#666', lw=0.8))

    # Strip overlay
    rng = np.random.default_rng(42)
    for i, cp in enumerate(cps):
        sub = long[long['checkpoint'] == cp]
        for _, r in sub.iterrows():
            ax.scatter(i + rng.uniform(-0.22, 0.22), r['abs_d_wells'],
                        s=24,
                        marker=MODEL_MARKERS[r['model']],
                        facecolors=COUNTRY_COLORS.get(r['country'], '#999'),
                        edgecolors='white', linewidth=0.35, alpha=0.75,
                        zorder=3)

    # Annotate median + max per checkpoint
    for i, cp in enumerate(cps):
        vals = long[long['checkpoint'] == cp]['abs_d_wells']
        med = vals.median(); mx = vals.max()
        ax.text(i, vals.max() * 1.05,
                f'median |Δ| = {med:.0f}\nmax |Δ| = {int(mx)}',
                ha='center', va='bottom', fontsize=9,
                color='#333',
                bbox=dict(boxstyle='round,pad=0.3',
                          facecolor='#fafafa', edgecolor='#bbb',
                          lw=0.5))

    ax.set_xticks(range(len(cps)))
    ax.set_xticklabels([f'@ {cp}' for cp in cps], fontsize=10.5)
    ax.set_ylabel('|Δ total_wells|  (descending − ascending)',
                  fontsize=11)
    ax.set_title(
        '(b) |Δ total_wells| at 2050, 2100, and 2179',
        fontsize=12, fontweight='bold', pad=8, loc='left')
    ax.grid(True, axis='y', linestyle=':', lw=0.6, alpha=0.4)
    # Headroom for the median/max annotation labels
    ax.set_ylim(0, long['abs_d_wells'].max() * 1.25)


def make_ordering_comparison_figure(deltas, output_path,
                                      selected_cases, model_markers,
                                      country_colors):
    """Render the 2-panel Q3 ordering-comparison figure.

    Parameters
    ----------
    deltas : DataFrame
        Per-case desc − asc table produced by `_build_deltas_table`.
    output_path : Path
        Stem path; saved as both .pdf and .png.
    """
    fig = plt.figure(figsize=(18, 10.5))
    gs = fig.add_gridspec(
        1, 3, width_ratios=[1.3, 1.0, 0.50],
        left=0.05, right=0.985, top=0.77, bottom=0.16, wspace=0.32)
    ax_a = fig.add_subplot(gs[0, 0])
    ax_b = fig.add_subplot(gs[0, 1])
    ax_legend = fig.add_subplot(gs[0, 2]); ax_legend.axis('off')

    _panel_a_severity(ax_a, deltas)
    _panel_b_wells(ax_b, deltas)

    # Subtitle band (between figure title and panels)
    # Three concise take-aways, one per line, italic
    n_diff_yrs = int((deltas['d_years_of_shortfall'] != 0).sum())
    n_diff_unmet = int((deltas['d_total_unmet_gt'].abs() > 0.1).sum())
    n_total = len(deltas)
    n_flips = int(deltas['pass_flip'].sum()) if 'pass_flip' in deltas.columns else 0
    subtitle_lines = [
        f'· Pass/fail verdict: {n_flips} / {n_total} cases flip pass ↔ fail under reordering.',
        f'· Failure severity shifts modestly in a minority of cases '
        f'({n_diff_yrs} / {n_total} differ on years_of_shortfall, '
        f'{n_diff_unmet} / {n_total} on |Δ unmet| > 0.1 Gt).',
        '· Operational well-fleet burden differs more systematically '
        '(see Panel b).',
    ]
    for i, line in enumerate(subtitle_lines):
        fig.text(
            0.05, 0.895 - i * 0.022, line,
            ha='left', va='top', fontsize=9.5, style='italic',
            color='#444')

    # Bottom caption (Panel b encoding)
    bottom_caption = (
        'Panel (b) encoding:  box = IQR  ·  whiskers = 1.5 × IQR  ·  '
        'dots = individual cases  ·  colour = country  ·  marker = model'
    )
    fig.text(
        0.5, 0.05, bottom_caption,
        ha='center', va='top', fontsize=10, style='italic', color='#444')

    # Country + model legends in the right sidebar
    country_handles = [
        plt.Line2D([0], [0], marker='o', linestyle='', markersize=8,
                    markerfacecolor=country_colors[c],
                    markeredgecolor='white', label=c)
        for c in selected_cases]
    model_handles = [
        plt.Line2D([0], [0], marker='o', linestyle='', markersize=8,
                    markerfacecolor='#888', markeredgecolor='white',
                    label='Logistic'),
        plt.Line2D([0], [0], marker='^', linestyle='', markersize=8,
                    markerfacecolor='#888', markeredgecolor='white',
                    label='Gompertz'),
    ]
    # Legends sit in the lower 2/3 of the sidebar so they read as a
    # unified "encoding key" panel rather than a top-pinned legend.
    leg_c = ax_legend.legend(
        handles=country_handles, loc='center',
        bbox_to_anchor=(0.5, 0.70), title='Country',
        fontsize=9.5, title_fontsize=10.5, frameon=False)
    ax_legend.add_artist(leg_c)
    ax_legend.legend(
        handles=model_handles, loc='center',
        bbox_to_anchor=(0.5, 0.28), title='Model',
        fontsize=9.5, title_fontsize=10.5, frameon=False)

    fig.suptitle(
        'Ordering comparison — severity vs CAPEX-related burden '
        f'(descending − ascending, n = {n_total} cases)',
        fontsize=13.5, fontweight='bold', y=0.965)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(str(output_path) + '.pdf', dpi=180, bbox_inches='tight')
    fig.savefig(str(output_path) + '.png', dpi=180, bbox_inches='tight')
    plt.close(fig)
    print(f'  wrote {output_path.relative_to(OUTPUT_ROOT.parent)}.{{pdf,png}}')


print('Generating ordering-comparison figure ...')
# Re-read the deltas table from disk so this cell is independently
# runnable; falls back to recomputing if the file is missing
deltas_path = OUTPUT_ROOT / 'extended_data' / 'ordering_sensitivity_deltas.csv'
if deltas_path.exists():
    deltas = pd.read_csv(deltas_path)
else:
    deltas = _build_deltas_table(sf_df, cp_all)

make_ordering_comparison_figure(
    deltas=deltas,
    output_path=OUTPUT_ROOT / 'fig_ordering_comparison',
    selected_cases=SELECTED_CASES,
    model_markers=MODEL_MARKERS,
    country_colors=COUNTRY_COLORS,
)
print('\nOrdering comparison figure rendered.')


Generating ordering-comparison figure ...


  wrote step2_screening/fig_ordering_comparison.{pdf,png}

Ordering comparison figure rendered.


## Figure 4: Per-country combined profile

3-row × 2-column layout per country × scenario:
- Top row: annual injection rate, stacked basin fill under the demand curve (Logistic | Gompertz)
- Middle row: portfolio proportion bar @2050, share of each basin, total wells, utilisation
- Bottom row: portfolio proportion bar @2100, same structure

Colour-coded legend with square markers matches basin fill.
Shortfall shaded red where demand exceeds delivery.

In [8]:
def draw_proportion_bar(ax, bm_cp, basin_colors, checkpoint_year, show_ylabel=True):
    """Draw a stacked horizontal proportion bar for one checkpoint."""
    if bm_cp.empty or bm_cp['allocated_rate_mt_yr'].sum() == 0:
        ax.text(0.5, 0.5, f'No active basins @{checkpoint_year}',
                transform=ax.transAxes, ha='center', va='center',
                fontsize=11, color='#999')
        ax.set_xlim(0, 1)
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        return

    # Sort by step_rank (allocation order)
    active = bm_cp[bm_cp['allocated_rate_mt_yr'] > 0].sort_values('step_rank')
    total_rate = active['allocated_rate_mt_yr'].sum()
    total_wells = int(active['wells_used'].sum())
    max_feas_total = bm_cp['max_feasible_rate_mt_yr'].sum()
    overall_util = total_rate / max_feas_total if max_feas_total > 0 else 0

    # Show all active basins (no "Others" aggregation)
    show_basins = list(active.iterrows())
    show_others = None

    # Draw stacked proportion bar
    left = 0.0
    bar_height = 0.6
    for _, row in show_basins:
        share = row['allocated_rate_mt_yr'] / total_rate
        color = basin_colors.get(row['basin'], '#999')
        ax.barh(0, share, left=left, height=bar_height,
                color=color, edgecolor='white', lw=1.0, alpha=0.88)

        # Label inside if wide enough
        if share > 0.08:
            name = row['basin']
            if len(name) > 12:
                name = name[:11] + '..'
            label = f'{name}\n{share:.0%}'
            ax.text(left + share / 2, 0, label, ha='center', va='center',
                    fontsize=7.5, fontweight='bold', color='white')
        left += share

    # (No 'Others' aggregation; all active basins shown individually above)

    # Summary annotation on the right
    ax.text(1.02, 0.35, f'delivered = min(demand, capacity)',
            transform=ax.get_yaxis_transform(), va='center', fontsize=7,
            color='#999', style='italic')
    rate_label = f'{total_rate:,.1f}' if total_rate < 10 else f'{total_rate:,.0f}'
    ax.text(1.02, 0, f'{rate_label} Mt/yr  |  {total_wells:,} wells  |  util {overall_util:.0%}',
            transform=ax.get_yaxis_transform(), va='center', fontsize=10,
            fontweight='bold', color='#333')

    ax.set_xlim(0, 1)
    ax.set_yticks([])
    if show_ylabel:
        ax.set_ylabel(f'@{checkpoint_year}', fontsize=12, fontweight='bold', rotation=0,
                      labelpad=45, va='center')
    ax.set_xticks([])
    for spine in ['top', 'right', 'bottom']:
        ax.spines[spine].set_visible(False)
    ax.spines['left'].set_visible(False)


def plot_country_combined(country, scenario, alloc_order, screened_df,
                          basin_alloc_list, bm_df, output_path):
    """Combined 3-row × 2-col: top = stacked rate, mid+bot = proportion bars."""
    output_path.parent.mkdir(parents=True, exist_ok=True)

    fig, axes = plt.subplots(3, 2, figsize=(28, 15),
                              gridspec_kw={'height_ratios': [1.4, 0.3, 0.3]},
                              constrained_layout=False)
    fig.subplots_adjust(hspace=0.30, wspace=0.45, left=0.06, right=0.82,
                        top=0.92, bottom=0.04)

    # Shared basin colour map
    all_basins_ordered = []
    for mdl in MODELS:
        ba_match = [b for b in basin_alloc_list
                    if not b.empty
                    and (b['country'] == country).any()
                    and (b['scenario'] == scenario).any()
                    and (b['model'] == mdl).any()
                    and (b['allocation_order'] == alloc_order).any()]
        if ba_match:
            ordering = ba_match[0].groupby('basin')['step'].min().sort_values()
            for name in ordering.index:
                if name not in all_basins_ordered:
                    all_basins_ordered.append(name)

    cmap_basins = plt.get_cmap('turbo')
    basin_colors = {name: cmap_basins(i / max(len(all_basins_ordered) - 1, 1))
                   for i, name in enumerate(all_basins_ordered)}

    for col_idx, mdl in enumerate(MODELS):
        # Top: Stacked injection rate
        ax_rate = axes[0, col_idx]

        sc = screened_df[
            (screened_df['country'] == country) &
            (screened_df['scenario'] == scenario) &
            (screened_df['model'] == mdl) &
            (screened_df['allocation_order'] == alloc_order)
        ].sort_values('year')

        if sc.empty:
            ax_rate.text(0.5, 0.5, f'No data\n({mdl})', transform=ax_rate.transAxes,
                         ha='center', va='center', fontsize=16, color='#999')
            ax_rate.set_title(mdl, fontsize=15, fontweight='bold')
            axes[1, col_idx].set_visible(False)
            axes[2, col_idx].set_visible(False)
            continue

        years = sc['year'].to_numpy()
        raw = sc['raw_rate_mt_yr'].to_numpy()
        yr_max = int(years.max())

        ba_match = [b for b in basin_alloc_list
                    if not b.empty
                    and (b['country'] == country).any()
                    and (b['scenario'] == scenario).any()
                    and (b['model'] == mdl).any()
                    and (b['allocation_order'] == alloc_order).any()]
        ba = ba_match[0] if ba_match else pd.DataFrame()

        legend_entries = []

        if not ba.empty:
            ordering = ba.groupby('basin').agg(
                step=('step', 'first'),
                rate=('allocated_rate_mt_yr', 'first'),
                wells=('wells_used', 'first'),
                start=('start_year', 'first'),
                end=('end_year', 'first'),
            ).sort_values('step')

            cumulative_bottom = np.zeros(len(years))
            for basin_name, brow in ordering.iterrows():
                contrib = np.zeros(len(years))
                active = (years >= brow['start']) & (years <= brow['end'])
                contrib[active] = brow['rate']

                total_with_this = cumulative_bottom + contrib
                clipped_top = np.minimum(total_with_this, raw)
                visible_contrib = np.maximum(clipped_top - cumulative_bottom, 0)

                color = basin_colors.get(basin_name, '#999')
                ax_rate.fill_between(years, cumulative_bottom,
                                     cumulative_bottom + visible_contrib,
                                     color=color, alpha=0.85,
                                     edgecolor='white', lw=0.3)
                ax_rate.plot(years[active],
                            (cumulative_bottom + visible_contrib)[active],
                            color='white', lw=0.5, alpha=0.6)
                cumulative_bottom += visible_contrib

                if brow['rate'] > 0:
                    rate_str = f"{brow['rate']:.1f}" if brow['rate'] >= 1 else f"{brow['rate']:.2f}" if brow['rate'] >= 0.01 else f"{brow['rate']:.3f}"
                    legend_entries.append((basin_name, rate_str, int(brow['wells']), color))

        # Demand curve
        ax_rate.plot(years, raw, 'k-', lw=2.5, zorder=10)

        # Shortfall shading
        screened_vals = sc['screened_rate_mt_yr'].to_numpy()
        ax_rate.fill_between(years, screened_vals, raw,
                             where=raw > screened_vals + 0.01,
                             color='#e15759', alpha=0.2, zorder=5)

        for cp in [2050, 2100, 2179]:
            if cp <= yr_max:
                ax_rate.axvline(cp, color='#888', ls='--', lw=1, alpha=0.5, zorder=8)

        ax_rate.set_xlim(2030, yr_max)
        ax_rate.set_xticks([2030, 2050, 2100, 2150, yr_max])
        ax_rate.set_xticklabels(['2030', '2050', '2100', '2150', str(yr_max)], fontsize=11)
        ax_rate.set_xlabel('Year', fontsize=12)
        if col_idx == 0:
            ax_rate.set_ylabel('Injection rate (Mt/yr)', fontsize=12)
        ax_rate.set_title(mdl, fontsize=15, fontweight='bold')

        # Colour legend with squares
        if legend_entries:
            legend_y = 0.97
            for name, rate_str, wells, color in legend_entries:
                ax_rate.text(1.01, legend_y, '\u25A0',
                            transform=ax_rate.transAxes, fontsize=11,
                            color=color, va='top')
                ax_rate.text(1.04, legend_y, f'{name} ({rate_str}, {wells:,}w)',
                            transform=ax_rate.transAxes, fontsize=7.5,
                            va='top', color='#333')
                legend_y -= 0.050

        # Demand + shortfall legend
        ax_rate.plot([], [], 'k-', lw=2.5, label='Demand')
        if any(raw > screened_vals + 0.01):
            ax_rate.fill_between([], [], [], color='#e15759', alpha=0.2, label='Shortfall')
        ax_rate.legend(loc='upper left', fontsize=9, frameon=True,
                      facecolor='white', edgecolor='#ddd')

        # (NEW Path D-full) Whole-period shortfall annotation strip
        # Read the whole-period shortfall row for this case if present
        sfw = sf_df[(sf_df['country'] == country) &
                    (sf_df['scenario'] == scenario) &
                    (sf_df['model'] == mdl) &
                    (sf_df['allocation_order'] == alloc_order)]
        if not sfw.empty and not bool(sfw.iloc[0]['whole_period_pass']):
            r0 = sfw.iloc[0]
            fy = int(r0['first_shortfall_year'])
            ly = int(r0['last_shortfall_year'])
            my = int(r0['max_gap_year'])
            mg = float(r0['max_gap_rate_mt_yr'])
            tu = float(r0['total_unmet_volume_gt'])
            ys = int(r0['years_of_shortfall'])
            y_top = ax_rate.get_ylim()[1]
            band_y_lo = y_top * 0.93
            band_y_hi = y_top * 0.97
            # translucent yellow band marking the shortfall window
            ax_rate.fill_between([fy, ly], band_y_lo, band_y_hi,
                                  color='#e15759', alpha=0.35, zorder=4)
            # red dashed vertical at max_gap_year
            ax_rate.axvline(my, color='#c0392b', lw=1.2,
                            linestyle='--', alpha=0.85, zorder=4)
            # compact annotation top-right
            ax_rate.text(ly + 1, band_y_hi,
                         f'{ys}y shortfall · '
                         f'{tu:.0f} Gt unmet · '
                         f'peak {mg:.0f} Mt/yr @{my}',
                         fontsize=8.5, color='#c0392b',
                         ha='left', va='top', zorder=5,
                         bbox=dict(boxstyle='round,pad=0.25',
                                   facecolor='white',
                                   edgecolor='#c0392b',
                                   alpha=0.9, linewidth=0.8))
        elif not sfw.empty and bool(sfw.iloc[0]['whole_period_pass']):
            y_top = ax_rate.get_ylim()[1]
            ax_rate.text(2035, y_top * 0.95,
                         '✓ whole-period pass',
                         fontsize=9, color='#1a9850',
                         fontweight='bold',
                         bbox=dict(boxstyle='round,pad=0.25',
                                   facecolor='white',
                                   edgecolor='#1a9850',
                                   alpha=0.9, linewidth=0.8))

        # Middle + Bottom: Proportion bars @2050 and @2100
        for row_idx, cp in [(1, 2050), (2, 2100)]:
            ax_bar = axes[row_idx, col_idx]
            bm_cp = bm_df[
                (bm_df['country'] == country) &
                (bm_df['scenario'] == scenario) &
                (bm_df['model'] == mdl) &
                (bm_df['allocation_order'] == alloc_order) &
                (bm_df['checkpoint'] == cp)
            ]
            draw_proportion_bar(ax_bar, bm_cp, basin_colors, cp,
                               show_ylabel=(col_idx == 0))

    # Within-profile y-axis sync (Logistic <-> Gompertz only)
    # Use the max raw demand across the two MODELS for this specific
    # (country, scenario, alloc_order). This keeps each profile readable
    # while making L and G directly comparable in height, without the
    # "small scenario dwarfed by max scenario" problem that comes from
    # syncing across all scenarios for a country.
    profile_data = screened_df[
        (screened_df['country'] == country) &
        (screened_df['scenario'] == scenario) &
        (screened_df['allocation_order'] == alloc_order)
    ]
    if not profile_data.empty:
        y_max_profile = float(profile_data['raw_rate_mt_yr'].max()) * 1.06
        if np.isfinite(y_max_profile) and y_max_profile > 0:
            for col in (0, 1):
                axes[0, col].set_ylim(0, y_max_profile)

    fig.suptitle(f'{country} | {scenario} | {ORDER_DIR_MAP[alloc_order]}',
                 fontsize=17, fontweight='bold')

    for fmt in ['pdf', 'png']:
        fig.savefig(str(output_path).replace('.pdf', f'.{fmt}'),
                   dpi=250, bbox_inches='tight')
    plt.close(fig)


# Generate Fig 4 for all combinations
for alloc_order in ALLOCATION_ORDERS:
    order_label = ORDER_DIR_MAP[alloc_order]
    prof_dir = OUTPUT_ROOT / order_label / 'figures' / 'country_profiles'
    prof_dir.mkdir(parents=True, exist_ok=True)

    for scenario in SCENARIOS:
        for case in SELECTED_CASES:
            has_rate = not screened_all[
                (screened_all['country'] == case) &
                (screened_all['scenario'] == scenario) &
                (screened_all['allocation_order'] == alloc_order)
            ].empty
            has_basin = not bm_all[
                (bm_all['country'] == case) &
                (bm_all['scenario'] == scenario) &
                (bm_all['allocation_order'] == alloc_order)
            ].empty
            if not (has_rate or has_basin):
                continue
            out_path = prof_dir / f'{slugify(case)}_{scenario}_profile.pdf'
            plot_country_combined(case, scenario, alloc_order,
                                  screened_all, all_basin_alloc, bm_all, out_path)

    n_figs = len(list(prof_dir.glob('*_profile.pdf')))
    print(f'  {order_label}: {n_figs} combined profile figures')


  descending: 76 combined profile figures


  ascending: 76 combined profile figures


## Output summary (Path D-full, V8 paper-ready package)

The screening package follows the three-question structure described
in the part-2 README.

### Q1: Can the basin portfolio match the growth curve?

* `case_model_shortfall_windows.csv` (per ordering): primary verdict
  table, one row per case with `whole_period_pass`, `years_of_shortfall`,
  `total_unmet_volume_gt`, etc.
* Figure 3: 2×2 whole-period feasibility scorecards
  (asc/desc × Logistic/Gompertz) at
  `output/step2_screening/fig3_dashboard.{pdf,png}`.

### Q2: Why does each case pass or fail?

* Figure 4 country profiles (per ordering) under
  `output/step2_screening/{ordering}/figures/country_profiles/`.
  Stacked basin-by-basin rate curves with demand overlaid, plus
  2050 and 2100 basin-share proportion bars.

### Q3: Does ordering change feasibility or just CAPEX?

* `fig_ordering_comparison.{pdf,png}` at the top of
  `output/step2_screening/`. Two panels:
  (a) per-case `Δ total_unmet_gt`,
  (b) `|Δ total_wells|` distribution at 2050 / 2100 / 2179.
* `extended_data/ordering_sensitivity_deltas.csv`: full per-case
  audit trail (150 rows × all Δ fields).

### Files removed in earlier cleanup

* `case_model_basin_service.csv` and the Fig 5 basin-service timelines.
* `delay_sensitivity.csv`, a legacy checkpoint-based diagnostic.
* Single-ordering dashboard variants are not part of the active paper package;
  Q1 is reported by the 2×2 scorecards and Q3 by the dedicated
  ordering-comparison figure.